# Análisis de Vuelos 2015 — Notebook de Analítica

Este notebook ejecuta las consultas SQL (P1–P5 y W1–W3) sobre PostgreSQL y Athena,
realiza una regresión OLS con `statsmodels` y un pronóstico de series de tiempo
con `StatsForecast`.

**Fuentes de datos:**
- PostgreSQL Read Replica → P1–P5, W1, W3
- Athena `flights_silver.flights_monthly` → W2, StatsForecast
- Athena `flights_gold.vuelos_analitica` → Regresión OLS

## Instalación de dependencias

Ejecutar una sola vez en SageMaker Studio.

In [ ]:
!pip install awswrangler statsmodels statsforecast plotly great-tables psycopg2-binary sqlalchemy -q

## Configuración

Reemplaza los placeholders con los valores del output de CloudFormation.

In [ ]:
import boto3
import json
import pandas as pd
import numpy as np
import awswrangler as wr
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from sqlalchemy import create_engine, text
from great_tables import GT, loc, style, html

# ── Configuración ──────────────────────────────────────────────────────────
REPLICA_ENDPOINT = "<RdsReplicaEndpoint>"          # Output: RdsReplicaEndpoint
BUCKET           = "itam-analytics-andrea-2026"
SECRET_NAME      = "itam/rds/flights/credentials"
REGION           = "us-east-1"
S3_OUTPUT        = f"s3://{BUCKET}/athena-results/"
DB_PORT          = 5432
DB_NAME          = "flights"

print("Configuracion lista")

## Conexión a PostgreSQL (Read Replica)

Las credenciales se recuperan desde AWS Secrets Manager — nunca se hardcodean.

In [ ]:
client = boto3.client("secretsmanager", region_name=REGION)
secret = client.get_secret_value(SecretId=SECRET_NAME)
creds  = json.loads(secret["SecretString"])

engine = create_engine(
    f"postgresql+psycopg2://{creds['username']}:{creds['password']}"
    f"@{REPLICA_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Conexion exitosa a PostgreSQL Read Replica")

---
## P1 — Top 10 rutas con mayor número de vuelos

¿Cuáles son las 10 rutas (origen → destino) con mayor número de vuelos?

In [ ]:
query_p1 = """
SELECT
    origin_airport,
    destination_airport,
    origin_airport || ' -> ' || destination_airport AS ruta,
    COUNT(*) AS total_vuelos
FROM flights
GROUP BY origin_airport, destination_airport
ORDER BY total_vuelos DESC
LIMIT 10
"""

df_p1 = pd.read_sql(query_p1, engine)
df_p1

In [ ]:
fig_p1 = px.bar(
    df_p1,
    x="total_vuelos",
    y="ruta",
    orientation="h",
    title="P1 — Top 10 rutas con mayor número de vuelos (2015)",
    labels={"total_vuelos": "Total de vuelos", "ruta": "Ruta"},
    color="total_vuelos",
    color_continuous_scale="Blues",
    width=800, height=450,
)
fig_p1.update_layout(
    yaxis={"categoryorder": "total ascending"},
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(showgrid=True, gridcolor="lightgrey"),
    coloraxis_showscale=False,
)
fig_p1.show()

---
## P2 — Top 5 aerolíneas con mayor porcentaje de cancelaciones

In [ ]:
query_p2 = """
SELECT
    f.airline,
    a.airline AS airline_name,
    COUNT(*) AS total_vuelos,
    SUM(cancelled) AS total_cancelaciones,
    ROUND(100.0 * SUM(cancelled) / COUNT(*), 2) AS pct_cancelados
FROM flights f
JOIN airlines a ON f.airline = a.iata_code
GROUP BY f.airline, a.airline
ORDER BY pct_cancelados DESC
LIMIT 5
"""

df_p2 = pd.read_sql(query_p2, engine)
df_p2

In [ ]:
fig_p2 = px.bar(
    df_p2,
    x="pct_cancelados",
    y="airline_name",
    orientation="h",
    title="P2 — Top 5 aerolíneas con mayor % de cancelaciones (2015)",
    labels={"pct_cancelados": "% Cancelados", "airline_name": "Aerolínea"},
    color="pct_cancelados",
    color_continuous_scale="Reds",
    text="pct_cancelados",
    width=800, height=400,
)
fig_p2.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_p2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(showgrid=True, gridcolor="lightgrey"),
    coloraxis_showscale=False,
)
fig_p2.show()

---
## P3 — Vuelos cancelados por causa

A=Carrier, B=Weather, C=NAS, D=Security

In [ ]:
query_p3 = """
SELECT
    cancellation_reason,
    CASE cancellation_reason
        WHEN 'A' THEN 'Carrier'
        WHEN 'B' THEN 'Weather'
        WHEN 'C' THEN 'NAS'
        WHEN 'D' THEN 'Security'
    END AS causa,
    COUNT(*) AS total_cancelados
FROM flights
WHERE cancellation_reason IS NOT NULL
GROUP BY cancellation_reason
ORDER BY total_cancelados DESC
"""

df_p3 = pd.read_sql(query_p3, engine)
df_p3

In [ ]:
(
    GT(df_p3[["causa", "cancellation_reason", "total_cancelados"]])
    .tab_header(
        title="P3 — Vuelos cancelados por causa",
        subtitle="Vuelos domésticos EE.UU. 2015"
    )
    .cols_label(
        causa=html("Causa"),
        cancellation_reason=html("Código"),
        total_cancelados=html("Total cancelados"),
    )
    .fmt_integer(columns="total_cancelados")
    .tab_style(
        style=style.fill(color="#fce4ec"),
        locations=loc.body(rows=[0]),
    )
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(columns="total_cancelados"),
    )
)

---
## P4 — Retraso promedio de salida por mes

Solo vuelos no cancelados con retraso de salida positivo.

In [ ]:
query_p4 = """
SELECT
    month,
    ROUND(AVG(departure_delay), 2) AS avg_departure_delay
FROM flights
WHERE cancelled = 0
  AND departure_delay > 0
GROUP BY month
ORDER BY month
"""

df_p4 = pd.read_sql(query_p4, engine)
df_p4

In [ ]:
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df_p4["mes_nombre"] = df_p4["month"].apply(lambda m: meses[m - 1])

fig_p4 = px.line(
    df_p4,
    x="mes_nombre",
    y="avg_departure_delay",
    markers=True,
    title="P4 — Retraso promedio de salida por mes (vuelos retrasados, 2015)",
    labels={"avg_departure_delay": "Retraso promedio (min)", "mes_nombre": "Mes"},
    width=800, height=400,
)
fig_p4.update_traces(line_color="#1565c0", marker=dict(size=8))
fig_p4.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=True, gridcolor="lightgrey"),
    xaxis=dict(showgrid=False),
)
fig_p4.show()

---
## P5 — Top 10 aeropuertos con más minutos de retraso por clima

In [ ]:
query_p5 = """
SELECT
    origin_airport,
    ROUND(SUM(weather_delay)) AS total_weather_delay_min
FROM flights
WHERE weather_delay IS NOT NULL
GROUP BY origin_airport
ORDER BY total_weather_delay_min DESC
LIMIT 10
"""

df_p5 = pd.read_sql(query_p5, engine)
df_p5

In [ ]:
fig_p5 = px.bar(
    df_p5,
    x="total_weather_delay_min",
    y="origin_airport",
    orientation="h",
    title="P5 — Top 10 aeropuertos con más minutos de retraso por clima (2015)",
    labels={"total_weather_delay_min": "Minutos de retraso por clima", "origin_airport": "Aeropuerto"},
    color="total_weather_delay_min",
    color_continuous_scale="Blues",
    width=800, height=450,
)
fig_p5.update_layout(
    yaxis={"categoryorder": "total ascending"},
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(showgrid=True, gridcolor="lightgrey"),
    coloraxis_showscale=False,
)
fig_p5.show()

---
## W1 — Vuelo con mayor retraso de llegada por aerolínea (RANK)

Usando `RANK()` particionando por aerolínea, ordenando por `arrival_delay DESC NULLS LAST`.

In [ ]:
query_w1 = """
WITH ranked AS (
    SELECT
        f.airline,
        a.airline AS airline_name,
        f.flight_number,
        f.origin_airport,
        f.destination_airport,
        f.arrival_delay,
        RANK() OVER (
            PARTITION BY f.airline
            ORDER BY f.arrival_delay DESC NULLS LAST
        ) AS rk
    FROM flights f
    JOIN airlines a ON f.airline = a.iata_code
)
SELECT airline, airline_name, flight_number,
       origin_airport, destination_airport, arrival_delay
FROM ranked
WHERE rk = 1
ORDER BY arrival_delay DESC
"""

df_w1 = pd.read_sql(query_w1, engine)
df_w1

In [ ]:
(
    GT(df_w1)
    .tab_header(
        title="W1 — Vuelo con mayor retraso de llegada por aerolínea",
        subtitle="Rank 1 por aerolínea — Vuelos 2015"
    )
    .cols_label(
        airline=html("Código"),
        airline_name=html("Aerolínea"),
        flight_number=html("Vuelo"),
        origin_airport=html("Origen"),
        destination_airport=html("Destino"),
        arrival_delay=html("Retraso llegada<br>(min)"),
    )
    .fmt_number(columns="arrival_delay", decimals=0)
    .tab_style(
        style=style.fill(color="#fff3e0"),
        locations=loc.body(rows=[0]),
    )
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(columns="arrival_delay"),
    )
)

---
## W2 — Variación mes a mes en total de vuelos (LAG)

> Esta query se ejecuta sobre `flights_silver.flights_monthly` **en Athena**,
> no sobre PostgreSQL. La carga de 500,000 filas en RDS solo cubre enero completo
> y parte de febrero — con 2 meses, `LAG()` produce una sola comparación.

In [ ]:
query_w2 = """
WITH mensual AS (
    SELECT
        month,
        SUM(total_flights) AS total_vuelos
    FROM flights_silver.flights_monthly
    GROUP BY month
),
con_lag AS (
    SELECT
        month,
        total_vuelos,
        LAG(total_vuelos) OVER (ORDER BY month) AS vuelos_mes_anterior,
        total_vuelos - LAG(total_vuelos) OVER (ORDER BY month) AS variacion_absoluta,
        ROUND(
            100.0 * (total_vuelos - LAG(total_vuelos) OVER (ORDER BY month))
            / NULLIF(LAG(total_vuelos) OVER (ORDER BY month), 0), 2
        ) AS variacion_pct
    FROM mensual
)
SELECT * FROM con_lag
ORDER BY month
"""

df_w2 = wr.athena.read_sql_query(
    sql=query_w2,
    database="flights_silver",
    s3_output=S3_OUTPUT,
    ctas_approach=False,
)
df_w2

In [ ]:
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
df_w2["mes_nombre"] = df_w2["month"].apply(lambda m: meses[m - 1])

fig_w2 = go.Figure()
fig_w2.add_trace(go.Bar(
    x=df_w2["mes_nombre"],
    y=df_w2["variacion_absoluta"],
    marker_color=["#ef5350" if v < 0 else "#42a5f5" for v in df_w2["variacion_absoluta"]],
    name="Variación vs mes anterior",
))
fig_w2.update_layout(
    title="W2 — Variación mes a mes en total de vuelos (2015)",
    xaxis_title="Mes",
    yaxis_title="Variación absoluta (vuelos)",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=True, gridcolor="lightgrey"),
    width=800, height=400,
)
fig_w2.show()

---
## W3 — Primeros 5 vuelos de LAX el 2015-01-01 (ROW_NUMBER)

Usando `ROW_NUMBER()` particionando por `(year, month, day, origin_airport)`.

In [ ]:
query_w3 = """
WITH numerados AS (
    SELECT
        flight_number,
        airline,
        destination_airport,
        scheduled_departure,
        ROW_NUMBER() OVER (
            PARTITION BY year, month, day, origin_airport
            ORDER BY scheduled_departure ASC
        ) AS rn
    FROM flights
    WHERE year = 2015
      AND month = 1
      AND day = 1
      AND origin_airport = 'LAX'
      AND cancelled = 0
)
SELECT flight_number, airline, destination_airport, scheduled_departure, rn
FROM numerados
WHERE rn <= 5
ORDER BY rn
"""

df_w3 = pd.read_sql(query_w3, engine)
df_w3

In [ ]:
(
    GT(df_w3)
    .tab_header(
        title="W3 — Primeros 5 vuelos desde LAX el 2015-01-01",
        subtitle="Ordenados por horario de salida programado"
    )
    .cols_label(
        flight_number=html("Vuelo"),
        airline=html("Aerolínea"),
        destination_airport=html("Destino"),
        scheduled_departure=html("Salida<br>programada"),
        rn=html("#"),
    )
    .tab_style(
        style=style.fill(color="#e3f2fd"),
        locations=loc.body(columns="rn"),
    )
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(columns=["flight_number", "rn"]),
    )
)

---
## Análisis Estadístico 8.1 — Regresión OLS

¿Qué factores explican el retraso de llegada?

**Variable objetivo:** `arrival_delay`  
**Features:** `departure_delay`, `distance`, `air_system_delay`, `airline_delay`,
`weather_delay`, `late_aircraft_delay`, `security_delay`

> Nota: los cinco componentes de retraso suman aproximadamente el retraso total.
> Su inclusión simultánea produce multicolinealidad — esto es intencional y parte del análisis.

In [ ]:
# Lee desde Gold en Athena
query_gold = """
SELECT
    arrival_delay,
    departure_delay,
    distance,
    air_system_delay,
    airline_delay,
    weather_delay,
    late_aircraft_delay,
    security_delay
FROM flights_gold.vuelos_analitica
WHERE cancelled = 0
  AND arrival_delay IS NOT NULL
"""

df_gold = wr.athena.read_sql_query(
    sql=query_gold,
    database="flights_gold",
    s3_output=S3_OUTPUT,
    ctas_approach=False,
)

print(f"Filas cargadas: {len(df_gold):,}")
df_gold.describe()

In [ ]:
from sklearn.metrics import mean_squared_error

features = [
    "departure_delay", "distance",
    "air_system_delay", "airline_delay", "weather_delay",
    "late_aircraft_delay", "security_delay",
]

df_ols = df_gold[features + ["arrival_delay"]].dropna()

X = sm.add_constant(df_ols[features])
y = df_ols["arrival_delay"]

modelo = sm.OLS(y, X).fit()
print(modelo.summary())

In [ ]:
y_pred = modelo.fittedvalues
rmse   = np.sqrt(mean_squared_error(y, y_pred))
r2     = modelo.rsquared

print(f"R²   = {r2:.4f}")
print(f"RMSE = {rmse:.4f} minutos")

In [ ]:
# Viz 1 — Coeficientes con intervalos de confianza al 95%
conf = modelo.conf_int(alpha=0.05)
df_coef = pd.DataFrame({
    "feature": features,
    "coef":    modelo.params[features].values,
    "ci_low":  conf.loc[features, 0].values,
    "ci_high": conf.loc[features, 1].values,
})

fig_coef = go.Figure()
fig_coef.add_trace(go.Bar(
    x=df_coef["coef"],
    y=df_coef["feature"],
    orientation="h",
    error_x=dict(
        type="data",
        symmetric=False,
        array=df_coef["ci_high"] - df_coef["coef"],
        arrayminus=df_coef["coef"] - df_coef["ci_low"],
    ),
    marker_color=["#ef5350" if c < 0 else "#42a5f5" for c in df_coef["coef"]],
))
fig_coef.update_layout(
    title="Viz 1 — Coeficientes OLS con IC 95%",
    xaxis_title="Coeficiente",
    yaxis_title="Feature",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    xaxis=dict(showgrid=True, gridcolor="lightgrey"),
    width=800, height=450,
)
fig_coef.show()

In [ ]:
# Viz 2 — Valores predichos vs. reales (muestra de 5,000 puntos)
sample_idx = np.random.choice(len(y), size=min(5000, len(y)), replace=False)

fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(
    x=y_pred.iloc[sample_idx],
    y=y.iloc[sample_idx],
    mode="markers",
    marker=dict(size=3, color="#1565c0", opacity=0.4),
    name="Observaciones",
))
lim = [min(y_pred.min(), y.min()), max(y_pred.max(), y.max())]
fig_pred.add_trace(go.Scatter(
    x=lim, y=lim, mode="lines",
    line=dict(color="red", dash="dash"),
    name="y = ŷ",
))
fig_pred.update_layout(
    title="Viz 2 — Valores predichos vs. reales",
    xaxis_title="ŷ (predicho)",
    yaxis_title="y (real)",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=700, height=500,
)
fig_pred.show()

In [ ]:
# Viz 3 — Residuos vs. valores predichos
residuos = y - y_pred

fig_res = go.Figure()
fig_res.add_trace(go.Scatter(
    x=y_pred.iloc[sample_idx],
    y=residuos.iloc[sample_idx],
    mode="markers",
    marker=dict(size=3, color="#1565c0", opacity=0.4),
    name="Residuos",
))
fig_res.add_hline(y=0, line_dash="dash", line_color="red")
fig_res.update_layout(
    title="Viz 3 — Residuos vs. valores predichos",
    xaxis_title="ŷ (predicho)",
    yaxis_title="Residuo (y - ŷ)",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=700, height=450,
)
fig_res.show()

In [ ]:
# Viz 4 — Q-Q plot de los residuos
import matplotlib.pyplot as plt
import scipy.stats as stats

fig_qq, ax = plt.subplots(figsize=(7, 5))
stats.probplot(residuos, dist="norm", plot=ax)
ax.set_title("Viz 4 — Q-Q plot de los residuos")
ax.get_lines()[0].set(markersize=2, alpha=0.3, color="#1565c0")
ax.get_lines()[1].set(color="red")
plt.tight_layout()
plt.show()

In [ ]:
print("""
Interpretación:

R² = {:.4f} — el modelo explica el {:.1f}% de la varianza en arrival_delay.
RMSE = {:.2f} minutos.

El gráfico de residuos vs. predichos muestra una estructura en forma de abanico
(heterocedasticidad): la varianza de los residuos aumenta con el nivel del retraso predicho.
Esto viola el supuesto de homocedasticidad de la regresión OLS.

El Q-Q plot confirma que los residuos tienen colas más pesadas que la normal teórica,
lo que afecta la validez de los intervalos de confianza reportados en el summary.

La multicolinealidad entre los componentes de retraso (airline_delay, weather_delay, etc.)
infla los errores estándar de esos coeficientes, haciéndolos menos interpretables
de forma individual.
""".format(r2, r2 * 100, rmse))

---
## Análisis Estadístico 8.2 — Pronóstico de Series de Tiempo

¿Cuántos vuelos habrá en los próximos meses?

**Datos:** `flights_silver.flights_monthly` — total de vuelos por mes (2015)  
**Train:** meses 1–9 | **Test:** meses 10–12 | **Forecast:** 6 meses adicionales  
**Modelos:** AutoETS, AutoARIMA, AutoTheta con `season_length=12`

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, AutoTheta

# Lee flights_monthly desde Silver en Athena
query_monthly = """
SELECT month, SUM(total_flights) AS total_flights
FROM flights_silver.flights_monthly
GROUP BY month
ORDER BY month
"""

df_monthly = wr.athena.read_sql_query(
    sql=query_monthly,
    database="flights_silver",
    s3_output=S3_OUTPUT,
    ctas_approach=False,
)
df_monthly

In [ ]:
# Prepara en formato StatsForecast: unique_id, ds, y
df_sf = pd.DataFrame({
    "unique_id": "vuelos_mensuales",
    "ds": pd.date_range(start="2015-01-01", periods=12, freq="MS"),
    "y":  df_monthly["total_flights"].values,
})

# Train: meses 1-9 | Test: meses 10-12
train = df_sf[df_sf["ds"] <= "2015-09-01"]
test  = df_sf[df_sf["ds"] >  "2015-09-01"]

print(f"Train: {len(train)} meses | Test: {len(test)} meses")
df_sf

In [ ]:
# Ajusta los tres modelos
# Con 9 puntos no hay suficientes datos para estimar estacionalidad de 12 meses
# — los modelos seleccionarán automáticamente especificaciones no estacionales.
sf = StatsForecast(
    models=[
        AutoETS(season_length=12),
        AutoARIMA(season_length=12),
        AutoTheta(season_length=12),
    ],
    freq="MS",
    n_jobs=-1,
)

sf.fit(train)
print("Modelos ajustados")

In [ ]:
# Horizonte: 9 pasos (3 test + 6 futuros) con IC al 90%
forecast = sf.predict(h=9, level=[90])
forecast = forecast.reset_index()
print("Columnas del forecast:", forecast.columns.tolist())
forecast

In [ ]:
# Modelos seleccionados automáticamente por cada algoritmo
for model in sf.models:
    print(f"{type(model).__name__}: {model}")

In [ ]:
# Viz 1 — Evaluación sobre el test set (meses 10-12)
forecast_test = forecast[forecast["ds"] <= "2015-12-01"]

fig_test = go.Figure()

# Datos reales (train + test)
fig_test.add_trace(go.Scatter(
    x=train["ds"], y=train["y"],
    mode="lines+markers", name="Train (real)",
    line=dict(color="black"),
))
fig_test.add_trace(go.Scatter(
    x=test["ds"], y=test["y"],
    mode="lines+markers", name="Test (real)",
    line=dict(color="black", dash="dot"),
))

colores = {"AutoETS": "#1565c0", "AutoARIMA": "#e65100", "AutoTheta": "#2e7d32"}
for modelo_nombre, color in colores.items():
    lo_col = f"{modelo_nombre}-lo-90"
    hi_col = f"{modelo_nombre}-hi-90"
    fig_test.add_trace(go.Scatter(
        x=pd.concat([forecast_test["ds"], forecast_test["ds"][::-1]]),
        y=pd.concat([forecast_test[hi_col], forecast_test[lo_col][::-1]]),
        fill="toself", fillcolor=color, opacity=0.15,
        line=dict(color="rgba(0,0,0,0)"), showlegend=False,
    ))
    fig_test.add_trace(go.Scatter(
        x=forecast_test["ds"], y=forecast_test[modelo_nombre],
        mode="lines+markers", name=modelo_nombre,
        line=dict(color=color),
    ))

fig_test.update_layout(
    title="Viz 1 — Evaluación en test set (Oct–Dic 2015) con IC 90%",
    xaxis_title="Mes", yaxis_title="Total de vuelos",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=True, gridcolor="lightgrey"),
    width=900, height=450,
)
fig_test.show()

In [ ]:
# Viz 2 — Pronóstico de 6 meses hacia adelante (Ene–Jun 2016)
forecast_future = forecast[forecast["ds"] > "2015-12-01"]

fig_fut = go.Figure()
fig_fut.add_trace(go.Scatter(
    x=df_sf["ds"], y=df_sf["y"],
    mode="lines+markers", name="2015 (real)",
    line=dict(color="black"),
))

for modelo_nombre, color in colores.items():
    lo_col = f"{modelo_nombre}-lo-90"
    hi_col = f"{modelo_nombre}-hi-90"
    fig_fut.add_trace(go.Scatter(
        x=pd.concat([forecast_future["ds"], forecast_future["ds"][::-1]]),
        y=pd.concat([forecast_future[hi_col], forecast_future[lo_col][::-1]]),
        fill="toself", fillcolor=color, opacity=0.15,
        line=dict(color="rgba(0,0,0,0)"), showlegend=False,
    ))
    fig_fut.add_trace(go.Scatter(
        x=forecast_future["ds"], y=forecast_future[modelo_nombre],
        mode="lines+markers", name=modelo_nombre,
        line=dict(color=color),
    ))

fig_fut.update_layout(
    title="Viz 2 — Pronóstico Ene–Jun 2016 con IC 90%",
    xaxis_title="Mes", yaxis_title="Total de vuelos",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=True, gridcolor="lightgrey"),
    width=900, height=450,
)
fig_fut.show()

In [ ]:
# Viz 3 — Comparación de MAE en el test set
from sklearn.metrics import mean_absolute_error

y_test = test["y"].values

mae_resultados = {}
for modelo_nombre in colores:
    y_hat = forecast_test[modelo_nombre].values
    mae_resultados[modelo_nombre] = mean_absolute_error(y_test, y_hat)

df_mae = pd.DataFrame(
    list(mae_resultados.items()),
    columns=["Modelo", "MAE"]
).sort_values("MAE")

print("MAE por modelo (test set Oct–Dic 2015):")
print(df_mae.to_string(index=False))

fig_mae = px.bar(
    df_mae,
    x="Modelo",
    y="MAE",
    title="Viz 3 — MAE por modelo en test set (vuelos/mes)",
    color="Modelo",
    color_discrete_map=colores,
    text="MAE",
    width=600, height=400,
)
fig_mae.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_mae.update_layout(
    showlegend=False,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=True, gridcolor="lightgrey"),
)
fig_mae.show()

In [ ]:
mejor_modelo = df_mae.iloc[0]["Modelo"]
mejor_mae    = df_mae.iloc[0]["MAE"]

print(f"""
Reporte:
- AutoETS   MAE = {mae_resultados['AutoETS']:,.0f} vuelos/mes
- AutoARIMA MAE = {mae_resultados['AutoARIMA']:,.0f} vuelos/mes
- AutoTheta MAE = {mae_resultados['AutoTheta']:,.0f} vuelos/mes

Mejor modelo: {mejor_modelo} (MAE = {mejor_mae:,.0f} vuelos/mes)

Con solo 9 puntos de entrenamiento los modelos seleccionaron especificaciones no
estacionales (insuficiente para estimar un ciclo de 12 meses). Los intervalos de
confianza al 90% son amplios, lo que refleja alta incertidumbre en el pronóstico
con apenas un año de datos.
""")